# Vježbe 12 — Vremenske serije: Trend, Sezonalnost i LSTM
**Predmet:** Rudarenje podataka — I. godina — II ciklus — Softversko inženjerstvo  
**Asistent:** ass. mr. Narcisa Hadžajlić  
**Datum:** 11.5.2026.

---

## Šta smo do sada naučili
- **Vježbe 8–9:** MLP, aktivacijske funkcije, regularizacija
- **Vježbe 10:** CNN — prostorna struktura slika, konvolucija, pooling
- **Vježbe 11:** Transfer Learning — VGG16, ResNet50, feature extraction, fine-tuning

## Novi tip podataka: Vremenska serija

Do sada smo radili s podacima gdje **redosljed nije važan** te svaki uzorak je nezavisan. Npr. izmješati redove u Iris datasetu ne mijenja ništa.

**Vremenska serija** je fundamentalno drugačija i **redosljed je ključan**. Temperatura u 15h ovisi o temperaturi u 14h. Cijena dionice sutra ovisi o kretanju prethodnih dana. Izmješati redove je = uništiti podatke.

### Plan:
1. Vizualizacija i razumijevanje vremenskih serija
2. Dekomponovanje: Trend + Sezonalnost + Reziduali
3. Zašto klasični MLP ne može uhvatiti vremensku zavisnost?
4. LSTM — mreža s memorijom
5. Predikcija temperaturne serije s LSTM-om
6. Zadatak — 4 boda

---
## 1. Šta je vremenska serija?

Svaki podatak ima **vremensku oznaku** i vrijednost. Između susjednih tačaka postoji **zavisnost** — prošlost utječe na budućnost.

### Tri komponente svake vremenske serije:

**Trend** — dugoročni smjer kretanja. Globalna temperatura raste. Populacija svijeta raste. Cijena nekih dionica raste, nekih pada.

**Sezonalnost (Seasonality)** — periodično ponavljanje u fiksnim intervalima. Prodaja sladoleda raste svako ljeto. Struja se troši više svaku zimu. Promet na web stranicama pada svaki vikend.

**Reziduali (šum)** — ono što ostane nakon što uklonimo trend i sezonalnost. Nepredvidivi događaji, greške mjerenja, nasumičnost.

```
Serija = Trend + Sezonalnost + Reziduali
```

### Zašto MLP nije dovoljan?

MLP prima fiksni vektor inputa i daje output. Nema pojam o tome šta se desilo **prije**. Ako mu date temperaturu od danas, ne zna da je juče bila veća i da se trend hladi. Svaki input tretira izolovano.

**LSTM (Long Short-Term Memory)** rješava ovo: ima **unutarnje stanje (hidden state)** koje se prenosi kroz vrijeme. Dok čita sekvencu, pamti relevantne informacije iz prošlosti.

---
## 2. Implementacija

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.get_logger().setLevel('ERROR')
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow: {tf.__version__}")
print("Sve uvezeno!")

### 2.1 Dataset — Temperature u Delhiju

Koristimo dataset dnevnih temperatura u Delhiju (2013–2017). Ima izražen godišnji sezonalni obrazac (ljeto/zima) i blagi trend i idealno je za ilustraciju svih koncepta.

In [ ]:
# Pokušaj s Kaggle-om, pa generiši sintetičke podatke ako ne radi
try:
    import kagglehub
    path = kagglehub.dataset_download("sumanthvrao/daily-climate-time-series-data")
    import glob
    csv_fajlovi = glob.glob(f"{path}/**/*.csv", recursive=True)
    print(f"Pronađeni CSV fajlovi: {csv_fajlovi}")
    df = pd.read_csv([f for f in csv_fajlovi if 'train' in f.lower()][0])
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date').sort_index()
    serija = df['meantemp'].dropna()
    print(f"Dataset učitan: {len(serija)} dnevnih tačaka")
    izvor = "Delhi Climate Dataset (Kaggle)"

except Exception as e:
    print(f"Kaggle nije dostupan ({e.__class__.__name__}). Generišem realistične sintetičke podatke...")

    # Sintetička serija s trendom + sezonalnošću + šumom
    # Identična struktura kao Delhi dataset
    datumi = pd.date_range(start='2013-01-01', end='2017-12-31', freq='D')
    n = len(datumi)
    t = np.arange(n)

    trend      = 0.003 * t                                          # blagi porast
    sezonalnost = 10 * np.sin(2 * np.pi * t / 365 - np.pi/2)       # godišnji ciklus
    sum_viseg   = 2 * np.sin(2 * np.pi * t / 180)                   # polugodišnji ciklus
    baza        = 25.0                                               # prosječna temp.
    sum_        = np.random.normal(0, 2, n)                         # šum

    temperature = baza + trend + sezonalnost + sum_viseg + sum_

    serija = pd.Series(temperature, index=datumi, name='meantemp')
    izvor  = "Sintetički podaci (ista struktura kao Delhi dataset)"
    print(f"Sintetička serija kreirana: {len(serija)} tačaka")

print(f"\nIzvor: {izvor}")
print(f"Period: {serija.index[0].date()} → {serija.index[-1].date()}")
print(f"Temp. opseg: {serija.min():.1f}°C — {serija.max():.1f}°C")
print(f"Prosjek: {serija.mean():.1f}°C")

In [ ]:
# Vizualizacija cijele serije
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(serija.index, serija.values, linewidth=0.8, color='steelblue', alpha=0.8)
ax.set_title(f'Dnevne temperature — {izvor}', fontsize=12)
ax.set_xlabel('Datum')
ax.set_ylabel('Temperatura (°C)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=45)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Šta vidimo na grafu?")
print("  → Periodično ponavljanje svake godine = SEZONALNOST")
print("  → Opći smjer kretanja kroz godine = TREND")
print("  → Sitne nepravilnosti = REZIDUALI (šum)")

### 2.2 Dekomponovanje — razdvajamo trend, sezonalnost i šum

`seasonal_decompose` iz `statsmodels` automatski razdvaja seriju na tri komponente. Model `additive` pretpostavlja da se komponente **sabiraju** (za multiplicativni model koristite `model='multiplicative'`).

In [ ]:
# Dekomponovanje
# period=365 → godišnja sezonalnost (dnevni podaci)
dekomponovanje = seasonal_decompose(
    serija,
    model='additive',
    period=365,
    extrapolate_trend='freq'  # popuni rubne vrijednosti trenda
)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

komponente = [
    (serija.values,                      'Originalna serija', 'steelblue'),
    (dekomponovanje.trend,               'Trend',             'coral'),
    (dekomponovanje.seasonal,            'Sezonalnost',       'seagreen'),
    (dekomponovanje.resid,               'Reziduali (šum)',   'purple'),
]

for ax, (data, naziv, boja) in zip(axes, komponente):
    ax.plot(serija.index, data, color=boja, linewidth=0.9)
    ax.set_ylabel(naziv, fontsize=10)
    ax.grid(True, alpha=0.2)

axes[0].set_title('Dekomponovanje vremenske serije: Originalna = Trend + Sezonalnost + Reziduali',
                  fontsize=11)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Kvantifikacija komponenti
print("Kvantifikacija komponenti:")
print(f"  Trend — opseg:       {dekomponovanje.trend.min():.2f}°C → {dekomponovanje.trend.max():.2f}°C")
print(f"  Sezonalnost — amp.:  ±{dekomponovanje.seasonal.std():.2f}°C (std)")
print(f"  Reziduali — std:     {dekomponovanje.resid.std():.2f}°C")
print(f"  → Sezonalnost objašnjava najveći dio varijance — LSTM mora ovo naučiti!")

### 2.3 Stacionarnost — ADF test

Statistički modeli preferiraju **stacionarne serije** — one bez trenda, s konstantnom varijancijom. ADF (Augmented Dickey-Fuller) test provjerava je li serija stacionarna.

LSTM ne zahtijeva strogo stacionarnost, ali razumijevanje ovog koncepta je važno za svaki ozbiljan projekt s vremenskim serijama.

In [ ]:
def adf_test(serija, naziv):
    rezultat = adfuller(serija.dropna())
    print(f"\nADF Test — {naziv}")
    print(f"  ADF statistika:  {rezultat[0]:.4f}")
    print(f"  p-vrijednost:    {rezultat[1]:.4f}")
    print(f"  Kritična vrij. (5%): {rezultat[4]['5%']:.4f}")
    if rezultat[1] < 0.05:
        print(f"  ✓ Serija JE stacionarna (p < 0.05)")
    else:
        print(f"  ✗ Serija NIJE stacionarna (p ≥ 0.05) — ima trend ili sezonalnost")

adf_test(serija, "Originalna temperatura")

# Diferenciranje — ukloni trend oduzimanjem prethodne vrijednosti
serija_diff = serija.diff().dropna()
adf_test(serija_diff, "Diferencirana serija (diff=1)")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6))
ax1.plot(serija.index, serija.values, color='steelblue', linewidth=0.8)
ax1.set_title('Originalna serija (može biti nestacionarna)')
ax1.grid(True, alpha=0.3)

ax2.plot(serija_diff.index, serija_diff.values, color='coral', linewidth=0.8)
ax2.set_title('Diferencirana serija (dan-na-dan promjena temperature)')
ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.4 LSTM — mreža s memorijom

**Intuicija:** MLP čita jednu tačku i daje odgovor. LSTM čita sekvencu tačaka i pamti kontekst.

Zamislite razliku između:
- MLP: "Temperatura je 15°C" → predvidi sutra
- LSTM: "Temperatura je bila 25°C, pa 22°C, pa 18°C, pa 15°C" → predvidi sutra

LSTM ima dvije vrste memorije:
- **Cell state (dugoročna)** — prenosi se kroz cijelu sekvencu, teško se mijenja
- **Hidden state (kratkoročna)** — output svakog koraka, brže se ažurira

### Sliding window — kako pripremamo podatke za LSTM

```
Originalna serija: [t1, t2, t3, t4, t5, t6, t7, t8 ...]

window_size = 30 (30 dana kao input)

Uzorak 1:  X=[t1..t30]   → y=t31
Uzorak 2:  X=[t2..t31]   → y=t32
Uzorak 3:  X=[t3..t32]   → y=t33
...
```

LSTM Input shape: **(batch_size, window_size, n_features)**

In [ ]:
# MinMaxScaler za vremenske serije
# Koristimo MinMax (0-1) umjesto Standard (mean=0, std=1)
# jer LSTM aktivacije rade bolje u opsegu 0-1

vrijednosti = serija.values.reshape(-1, 1).astype('float32')

scaler = MinMaxScaler(feature_range=(0, 1))
skalirano = scaler.fit_transform(vrijednosti).flatten()

print(f"Originalni opseg: {vrijednosti.min():.2f} — {vrijednosti.max():.2f}°C")
print(f"Skalirani opseg:  {skalirano.min():.2f} — {skalirano.max():.2f}")

# Train/test split — VAŽNO: ne smijemo miješati!
# Test = posljednjih 20% (hronološki)
split = int(len(skalirano) * 0.80)
train_data = skalirano[:split]
test_data  = skalirano[split:]

print(f"\nTrain: {len(train_data)} tačaka ({serija.index[0].date()} → {serija.index[split-1].date()})")
print(f"Test:  {len(test_data)} tačaka ({serija.index[split].date()} → {serija.index[-1].date()})")
print("\nKritično: split je hronološki — nikad nasumičan za vremenske serije!")

In [ ]:
def napravi_sekvence(data, window_size):
    """
    Pretvara 1D seriju u 2D array sekvenci za LSTM.
    Svaki uzorak: prethodnih window_size vrijednosti → predvidi sljedeću.
    """
    X, y = [], []
    for i in range(window_size, len(data)):
        X.append(data[i - window_size : i])  # prozor unazad
        y.append(data[i])                     # sljedeća vrijednost
    X = np.array(X)
    y = np.array(y)
    # LSTM treba 3D input: (uzorci, koraci_u_vremenu, features)
    X = X.reshape(X.shape[0], X.shape[1], 1)
    return X, y


WINDOW = 30  # koristimo 30 dana za predikciju sljedećeg

X_train, y_train = napravi_sekvence(train_data, WINDOW)
X_test,  y_test  = napravi_sekvence(test_data,  WINDOW)

print(f"Window size: {WINDOW} dana")
print(f"X_train shape: {X_train.shape}  ← (uzorci, koraci, features)")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"\nSvaki uzorak = {WINDOW} uzastopnih dana → predviđamo {WINDOW+1}. dan")

### 2.5 Gradimo LSTM model

In [ ]:
def napravi_lstm(window_size, lstm_jedinice=64):
    model = keras.Sequential([
        layers.Input(shape=(window_size, 1)),
        # Prvi LSTM layer — return_sequences=True jer sljedi drugi LSTM
        # return_sequences=True → vraća output ZA SVAKI korak u sekvenci
        # return_sequences=False → vraća samo ZADNJI output (default)
        layers.LSTM(lstm_jedinice, return_sequences=True),
        layers.Dropout(0.2),
        # Drugi LSTM layer — return_sequences=False jer sljedi Dense
        layers.LSTM(lstm_jedinice // 2, return_sequences=False),
        layers.Dropout(0.2),
        # Dense layeri za finalnu predikciju
        layers.Dense(32, activation='relu'),
        layers.Dense(1)  # regresija — jedna kontinualna vrijednost, bez aktivacije!
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',   # Mean Squared Error — standardni loss za regresiju
        metrics=['mae']  # Mean Absolute Error — lakše za interpretaciju
    )
    return model


lstm_model = napravi_lstm(WINDOW)
lstm_model.summary()

print("\nVažna razlika od klasifikacije:")
print("  → Output Dense(1) BEZ aktivacije = regresija (predviđamo broj, ne klasu)")
print("  → Loss je MSE, ne crossentropy")
print("  → Metrika je MAE (srednja apsolutna greška), ne accuracy")

In [ ]:
print("Treniranje LSTM-a...")

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,      # prepolovi learning rate
    patience=5,      # ako nema poboljšanja 5 epoha
    min_lr=1e-6
)

historija = lstm_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print(f"\nTrening završen nakon {len(historija.history['loss'])} epoha.")

### 2.6 Evaluacija — predikcija vs. stvarnost

In [ ]:
# Predikcije na test skupu
y_pred_scaled = lstm_model.predict(X_test, verbose=0).flatten()

# Inverz skaliranja — nazad u °C
y_pred_celsius = scaler.inverse_transform(
    y_pred_scaled.reshape(-1, 1)
).flatten()

y_test_celsius = scaler.inverse_transform(
    y_test.reshape(-1, 1)
).flatten()

# Metrike
mae  = mean_absolute_error(y_test_celsius, y_pred_celsius)
rmse = np.sqrt(mean_squared_error(y_test_celsius, y_pred_celsius))
mape = np.mean(np.abs((y_test_celsius - y_pred_celsius) / 
               (y_test_celsius + 1e-8))) * 100

print("Metrike na test skupu:")
print(f"  MAE  (Srednja apsolutna greška): {mae:.2f}°C")
print(f"  RMSE (Korijen srednje kv. gr.):  {rmse:.2f}°C")
print(f"  MAPE (Postotna greška):          {mape:.2f}%")
print(f"\n  Interpretacija: prosječna greška je ±{mae:.1f}°C")

In [ ]:
# Vizualizacija — stvarnost vs. predikcija
test_datumi = serija.index[split + WINDOW:]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Cijela serija s označenim test periodom
axes[0].plot(serija.index, serija.values,
             color='steelblue', linewidth=0.8, label='Originalna serija')
axes[0].axvspan(test_datumi[0], test_datumi[-1],
                alpha=0.15, color='red', label='Test period')
axes[0].set_title('Cijela serija — test period označen crvenom')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Zoom na test period
n_prikaz = min(365, len(test_datumi))  # prikazi max godinu dana
axes[1].plot(test_datumi[:n_prikaz], y_test_celsius[:n_prikaz],
             color='steelblue', linewidth=1.5, label='Stvarna temperatura', alpha=0.8)
axes[1].plot(test_datumi[:n_prikaz], y_pred_celsius[:n_prikaz],
             color='coral',     linewidth=1.5, label='LSTM predikcija',     alpha=0.8,
             linestyle='--')
axes[1].fill_between(test_datumi[:n_prikaz],
                     y_test_celsius[:n_prikaz],
                     y_pred_celsius[:n_prikaz],
                     alpha=0.2, color='red', label='Greška')
axes[1].set_title(f'Test period — Stvarno vs. Predviđeno (MAE={mae:.2f}°C)')
axes[1].set_ylabel('Temperatura (°C)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Kriva učenja
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(historija.history['loss'],     label='Train', color='coral',     linewidth=2)
ax1.plot(historija.history['val_loss'], label='Val',   color='steelblue', linewidth=2)
ax1.set_title('Loss (MSE) kroz epohe')
ax1.set_xlabel('Epoha')
ax1.set_ylabel('MSE')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(historija.history['mae'],     label='Train MAE', color='coral',     linewidth=2)
ax2.plot(historija.history['val_mae'], label='Val MAE',   color='steelblue', linewidth=2)
ax2.set_title('MAE kroz epohe')
ax2.set_xlabel('Epoha')
ax2.set_ylabel('MAE (skalirano)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.7 Usporedba: LSTM vs. Naivni model vs. MLP

**Naivni model** (baseline): predvidi da je sutra ista temperatura kao danas. Jednostavan, ali iznenađujuće dobar za kratkoročnu predikciju. Uvijek treba imati baseline za usporedbu!

In [ ]:
# Baseline: naivna predikcija (sutra = danas)
y_naive = y_test_celsius[:-1]  # shift za 1
y_true_naive = y_test_celsius[1:]

mae_naive  = mean_absolute_error(y_true_naive, y_naive)
rmse_naive = np.sqrt(mean_squared_error(y_true_naive, y_naive))

# MLP za usporedbu
X_train_mlp = X_train.reshape(X_train.shape[0], -1)
X_test_mlp  = X_test.reshape(X_test.shape[0], -1)

mlp_model = keras.Sequential([
    layers.Input(shape=(WINDOW,)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])
mlp_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
mlp_model.fit(
    X_train_mlp, y_train,
    epochs=50, batch_size=32,
    validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=0
)

y_pred_mlp_scaled = mlp_model.predict(X_test_mlp, verbose=0).flatten()
y_pred_mlp_celsius = scaler.inverse_transform(
    y_pred_mlp_scaled.reshape(-1, 1)
).flatten()

mae_mlp  = mean_absolute_error(y_test_celsius, y_pred_mlp_celsius)
rmse_mlp = np.sqrt(mean_squared_error(y_test_celsius, y_pred_mlp_celsius))

# Prikaz usporedbe
print("Usporedba modela:")
print("-" * 55)
print(f"{'Model':20s}  {'MAE (°C)':>10s}  {'RMSE (°C)':>10s}")
print("-" * 55)
for naziv, mae_v, rmse_v in [
    ('Naivni (sutra=danas)', mae_naive, rmse_naive),
    ('MLP (flatten prozor)', mae_mlp,   rmse_mlp),
    ('LSTM',                 mae,       rmse),
]:
    marker = " ← BEST" if mae_v == min(mae_naive, mae_mlp, mae) else ""
    print(f"{naziv:20s}  {mae_v:>10.3f}  {rmse_v:>10.3f}{marker}")

In [ ]:
# Vizualizacija usporedbe — kratki period
n = 90  # 90 dana za prikaz

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_datumi[:n], y_test_celsius[:n],
        color='black',     linewidth=2,   label='Stvarno',        alpha=0.9)
ax.plot(test_datumi[:n], y_pred_celsius[:n],
        color='coral',     linewidth=1.5, label=f'LSTM (MAE={mae:.2f}°C)',
        linestyle='--')
ax.plot(test_datumi[:n], y_pred_mlp_celsius[:n],
        color='steelblue', linewidth=1.5, label=f'MLP  (MAE={mae_mlp:.2f}°C)',
        linestyle=':')
ax.plot(test_datumi[1:n+1], y_naive[:n],
        color='seagreen',  linewidth=1.5, label=f'Naivni (MAE={mae_naive:.2f}°C)',
        linestyle='-.')

ax.set_title('Usporedba modela — prvih 90 dana test perioda', fontsize=12)
ax.set_ylabel('Temperatura (°C)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nKljučna opservacija:")
print("  Naivni model je iznenađujuće dobar za 1-korak predikciju")
print("  LSTM je bolji na dužim horizontima i pri naglim promjenama")
print("  MLP prati obrazac ali ne razumije vremenski slijed")

### 2.8 Predikcija u budućnost (multi-step)

Zanimljiviji slučaj: predvidi **N koraka unaprijed**, ne samo jedan. Koristimo **rekurzivnu predikciju** — svaka predikcija postaje input za sljedeću.

In [ ]:
def predvidi_n_koraka(model, poslednji_prozor, n_koraka, scaler):
    """
    Rekurzivna multi-step predikcija.
    poslednji_prozor: numpy array shape (window_size,) — skalirane vrijednosti
    """
    predikcije = []
    trenutni_prozor = poslednji_prozor.copy()

    for _ in range(n_koraka):
        # Pripremi input: (1, window_size, 1)
        x = trenutni_prozor.reshape(1, len(trenutni_prozor), 1)
        # Predvidi sljedeći korak
        pred = model.predict(x, verbose=0)[0, 0]
        predikcije.append(pred)
        # Pomjeri prozor: ukloni prvi element, dodaj predikciju
        trenutni_prozor = np.append(trenutni_prozor[1:], pred)

    # Inverz skaliranja
    predikcije_celsius = scaler.inverse_transform(
        np.array(predikcije).reshape(-1, 1)
    ).flatten()
    return predikcije_celsius


# Uzmimo zadnjih 30 dana iz test skupa kao polaznu tačku
poslednji_prozor = test_data[-WINDOW:]
N_BUDUCNOST = 60  # predvidi 60 dana unaprijed

predikcije_buducnost = predvidi_n_koraka(
    lstm_model, poslednji_prozor, N_BUDUCNOST, scaler
)

# Datumi za predikcije
zadnji_datum      = serija.index[-1]
buduci_datumi     = pd.date_range(
    start=zadnji_datum + pd.Timedelta(days=1),
    periods=N_BUDUCNOST, freq='D'
)

# Vizualizacija
fig, ax = plt.subplots(figsize=(14, 5))

# Posljednjih 90 dana poznatih podataka
ax.plot(serija.index[-90:], serija.values[-90:],
        color='steelblue', linewidth=2, label='Poznati podaci')

# Predikcije
ax.plot(buduci_datumi, predikcije_buducnost,
        color='coral', linewidth=2, linestyle='--',
        label=f'LSTM predikcija ({N_BUDUCNOST} dana)')

# Vertikalna linija na granici poznato/nepoznato
ax.axvline(zadnji_datum, color='gray', linestyle=':', linewidth=1.5)
ax.text(zadnji_datum, ax.get_ylim()[0] + 1, '  kraj poznatih\n  podataka',
        fontsize=9, color='gray')

ax.set_title(f'Multi-step predikcija — {N_BUDUCNOST} dana u budućnost', fontsize=12)
ax.set_ylabel('Temperatura (°C)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\nPredviđene temperature za narednih {N_BUDUCNOST} dana:")
print(f"  Min: {predikcije_buducnost.min():.1f}°C")
print(f"  Max: {predikcije_buducnost.max():.1f}°C")
print(f"  Prosjek: {predikcije_buducnost.mean():.1f}°C")
print("\nNapomena: greška se akumulira kod rekurzivne predikcije.")
print("Što dalje u budućnost, to manje pouzdana predikcija.")

---
## 3. Šta smo naučili danas

| Koncept | Šta znači |
|---|---|
| **Trend** | Dugoročni smjer kretanja serije |
| **Sezonalnost** | Periodično ponavljanje u fiksnim intervalima |
| **Reziduali** | Šum koji ostane nakon uklanjanja trenda i sezonalnosti |
| **Stacionarnost** | Serija bez trenda, konstantne statističke osobine |
| **Sliding window** | Pretvaranje serije u (X, y) parove za supervizovano učenje |
| **LSTM** | RNN s dugoročnom i kratkoročnom memorijom |
| **`return_sequences`** | `True` ako slijedi drugi LSTM, `False` ako slijedi Dense |
| **MinMaxScaler** | Skaliranje u (0,1) — preporučeno za LSTM |
| **Hronološki split** | Test skup = kraj serije, nikad nasumičan! |
| **MAE / RMSE** | Metrike za regresiju — greška u originalnim jedinicama |
| **Naivni baseline** | Predikcija sutra=danas — uvijek usporedite s ovim! |
| **Multi-step predikcija** | Rekurzivno predviđanje N koraka unaprijed |

---
## ZADATAK — 4 BODA

**Rok:** 13.05.2026. do 23:59

### Šta trebate uraditi:

1. **Pronađite time series dataset** na [kaggle.com](https://kaggle.com):
   - Preporučeno: `Bitcoin Historical Data`, `Air Quality UCI`, `Avocado Prices`, `Energy consumption`, `Store Sales Forecasting`
   - Mora imati vremensku kolonu i barem jednu numeričku vrijednost
   - Minimalno 500 tačaka

2. **Dekomponujte seriju** s `seasonal_decompose` i vizualizujte sve 4 komponente

3. **Istrenirajte LSTM** i prikažite:
   - Sliding window pripremu podataka
   - Kriva učenja
   - Predikcija vs. stvarnost na test skupu
   - MAE i RMSE

4. **Usporedite s naivnim modelom**: je li vaš LSTM bolji od "sutra=danas"?

5. **Napišite komentar:**
   - Koji dataset ste uzeli?
   - Koji window size ste koristili i zašto?
   - Da li LSTM pobijedi naivni model? Ako ne šta mislite zašto?


### Bodovanje:
- **1 bod** — Dekomponovanje i vizualizacija, hronološki split, LSTM treniran
- **1.5 bod** — MAE/RMSE prikazani, usporedba s naivnim modelom, vlastiti dataset
- **1.5 bod** — Komentar s razumijevanjem — posebno odgovor na pitanje o naivnom modelu